In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.callbacks import StdOutCallbackHandler

import os

load_dotenv()

chat = ChatOpenAI(
    base_url = os.getenv("OPENAI_BASE_URL"),
    api_key = os.getenv("OPENAI_API_KEY"), 
    model = os.getenv("OPENAI_MODEL_NAME", "gpt-5.1"),
    temperature = 0.1,
    streaming = True,
    callbacks=[StdOutCallbackHandler()]
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import BaseOutputParser

class QuizOutputParser(BaseOutputParser):
    def parse(self, text: str):
        items = text.strip().split("\n")
        return items
    
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", ""),
    ("human", "Summary this {topic} in 3 sentences.") 
])

summary_chain = summary_prompt | chat

quiz_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a quiz generator. Based on the {summary}, generate 1 multiple-choice quiz only. Do not need answer key for each question."),
    ("human", "{summary}") 
])

quiz_chain = quiz_prompt | chat

final_chain = {"summary": summary_chain} | quiz_chain | QuizOutputParser()

final_chain.invoke({"topic": "the history of the internet"})